In [25]:
import torch
from usta_model import UstaModel
from usta_tokenizer import UstaTokenizer

u_tokenizer = UstaTokenizer("tokenizer.json")

prompt = "the capital of united"

tokens = u_tokenizer.encode(prompt)
tokens.shape

torch.Size([7])

In [26]:
torch.manual_seed(1)
u_model = UstaModel(vocab_size=len(u_tokenizer.vocab), embedding_dim=4, context_length=32)

sentence_meanings_with_attention_context = u_model(tokens)
sentence_meanings_with_attention_context

tensor([[-1.5256, -0.7502, -0.6540, -1.6095],
        [-0.2092, -0.2013, -0.7509, -0.0189],
        [ 0.9326, -0.2774,  0.3166, -1.6980],
        [ 0.7698, -0.1935,  0.1223, -0.0585],
        [-0.1228,  0.3777,  1.0470, -0.1133],
        [-0.4315, -0.1780,  0.6491, -0.0958],
        [-0.1496,  1.1284,  0.2814,  1.3386]], grad_fn=<CopySlices>)

In [27]:
from transformers import Gemma3ForCausalLM

gemma_model = Gemma3ForCausalLM.from_pretrained("google/gemma-3-1b-it")
u_model, gemma_model

Loading weights: 100%|██████████| 340/340 [00:00<00:00, 1222.17it/s, Materializing param=model.norm.weight]                                


(UstaModel(
   (embedding): Embedding(64, 4)
   (pos_embedding): Embedding(32, 4)
 ),
 Gemma3ForCausalLM(
   (model): Gemma3TextModel(
     (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 1152, padding_idx=0)
     (layers): ModuleList(
       (0-25): 26 x Gemma3DecoderLayer(
         (self_attn): Gemma3Attention(
           (q_proj): Linear(in_features=1152, out_features=1024, bias=False)
           (k_proj): Linear(in_features=1152, out_features=256, bias=False)
           (v_proj): Linear(in_features=1152, out_features=256, bias=False)
           (o_proj): Linear(in_features=1024, out_features=1152, bias=False)
           (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
           (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
         )
         (mlp): Gemma3MLP(
           (gate_proj): Linear(in_features=1152, out_features=6912, bias=False)
           (up_proj): Linear(in_features=1152, out_features=6912, bias=False)
           (down_proj): Linear(in_features=6912, out_features=1152, b

![Attention Diagram](https://lena-voita.github.io/resources/lectures/seq2seq/transformer/qkv_attention_formula-min.png)

In [28]:
q_weights = torch.nn.Linear(4, 3, bias=False)
k_weights = torch.nn.Linear(4, 3, bias=False)
v_weights = torch.nn.Linear(4, 3, bias=False)

q_of_sentence = q_weights(sentence_meanings_with_attention_context)
k_of_sentence = k_weights(sentence_meanings_with_attention_context)
v_of_sentence = v_weights(sentence_meanings_with_attention_context)
print(q_weights.weight)

q_of_sentence.shape, k_of_sentence.shape, v_of_sentence.shape


Parameter containing:
tensor([[-0.3231,  0.3206,  0.1776,  0.1237],
        [-0.0991, -0.1560,  0.0848,  0.3869],
        [ 0.0716, -0.2364,  0.4967,  0.0107]], requires_grad=True)


(torch.Size([7, 3]), torch.Size([7, 3]), torch.Size([7, 3]))

In [29]:
k_of_sentence.shape

torch.Size([7, 3])

In [30]:
attention_scores = q_of_sentence @ k_of_sentence.T
attention_weights = torch.softmax(attention_scores / k_of_sentence.shape[-1] ** 0.5, dim=1)

context_vector = attention_weights @ v_of_sentence
context_vector

tensor([[-0.0893, -0.0388,  0.0322],
        [-0.1083, -0.0509,  0.0494],
        [-0.1654, -0.0923,  0.1070],
        [-0.1653, -0.0886,  0.0988],
        [-0.1269, -0.0566,  0.0440],
        [-0.1385, -0.0656,  0.0590],
        [-0.0841, -0.0279,  0.0086]], grad_fn=<MmBackward0>)

In [31]:
from plot_tokens import plot_tokens

u_sentences = [
  {
    "words": q_of_sentence.detach().numpy(),
    "labels": u_tokenizer.tokenize(prompt),
    "color": "blue",
  },
  {
    "words": k_of_sentence.detach().numpy(),
    "labels": u_tokenizer.tokenize(prompt),
    "color": "purple",
  },
  {
    "words": v_of_sentence.detach().numpy(),
    "labels": u_tokenizer.tokenize(prompt),
    "color": "orange",
  },
  {
    "words": context_vector.detach().numpy(),
    "labels": u_tokenizer.tokenize(prompt),
    "color": "green",
  },
]

plot_tokens(u_sentences, "Query, Key, Value and Context Vector Space")